In [ ]:
# Bibliotecas
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import re, unicodedata

# (Opcional) tamanho padrão de figuras
plt.rcParams["figure.figsize"] = (10, 5)

In [ ]:
import os, glob

# Nome do arquivo baixado do ambiente do curso.
# Se o seu arquivo tiver outro nome, troque aqui.
NOME_ARQUIVO = "2019_Viagem.csv"

# Procura o arquivo nos lugares mais comuns
candidatos = [
    NOME_ARQUIVO,
    os.path.join("dados", NOME_ARQUIVO),
    os.path.join("/content", NOME_ARQUIVO),                # Google Colab
    os.path.join("/content/drive/MyDrive", NOME_ARQUIVO),  # Colab + Google Drive
    os.path.expanduser(os.path.join("~", "Downloads", NOME_ARQUIVO)),
]
caminho = next((c for c in candidatos if os.path.exists(c)), None)

# Ainda nao achou? Procura qualquer CSV com "Viagem" no nome, aqui e um nivel abaixo
if caminho is None:
    achados = sorted(glob.glob("*Viagem*.csv") + glob.glob("*/*Viagem*.csv"))
    if achados:
        caminho = achados[0]
        print("Usando o arquivo encontrado:", caminho)

if caminho is None:
    print("Pasta atual:", os.getcwd())
    print("CSVs por perto:", sorted(glob.glob("*.csv") + glob.glob("*/*.csv")) or "nenhum")
    raise FileNotFoundError(
        f"Nao encontrei '{NOME_ARQUIVO}'. Envie o CSV para a pasta do notebook "
        "(no Colab: painel de arquivos a esquerda; no Jupyter no navegador: "
        "arraste o arquivo para a lista de arquivos) ou ajuste NOME_ARQUIVO acima."
    )

# O separador e ';' e o decimal e ',' (padrao brasileiro),
# e o arquivo vem codificado em latin-1.
print("Lendo:", caminho)
df = pd.read_csv(
    caminho,
    sep=";",
    decimal=",",
    encoding="latin-1",
    engine="python",
    quotechar='"',
    escapechar="\\",
    on_bad_lines="warn",
)
print("Carregado:", df.shape)

In [ ]:
# Checagens iniciais
print("Dimensões:", df.shape)
display(df.head(10))
print(df.info())

In [ ]:
import re, unicodedata
import pandas as pd

# 1) Veja como o Pandas leu os nomes (só para conferência)
print("Colunas originais:", list(df.columns))

# 2) Função para padronizar nomes de colunas
#    (tira acentos, troca símbolos por "_" e deixa tudo minúsculo)
def normalizar_coluna(s):
    s = str(s)
    s = unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode("ascii")
    s = re.sub(r"[^\w]+", "_", s)   # qualquer coisa que não é [A-Za-z0-9_] vira "_"
    s = re.sub(r"_+", "_", s)       # colapsa ___ em _
    return s.strip("_").lower()

df.columns = [normalizar_coluna(c) for c in df.columns]
print("Colunas normalizadas:", list(df.columns))

# 3) Função para achar coluna por palavras-chave
#    (todas as palavras devem aparecer no nome da coluna)
def achar_coluna(palavras):
    for c in df.columns:
        if all(p in c for p in palavras):
            return c
    return None

# 4) Descobrir as colunas relevantes automaticamente
col_data_bruta = (achar_coluna(["periodo", "data", "inicio"])
                  or achar_coluna(["data", "inicio"])
                  or achar_coluna(["inicio"]))  # mais tolerante
col_valor_passagens = (achar_coluna(["valor", "passag"])
                       or achar_coluna(["passag"]))  # "valor_passagens"
col_orgao = (achar_coluna(["nome", "orgao", "superior"])
             or achar_coluna(["orgao"]))
col_destino = achar_coluna(["destino"])  # "destinos"
col_idproc  = achar_coluna(["identificador", "processo", "viagem"])  # id do processo
col_situacao = achar_coluna(["situacao"])

print("Mapeamento encontrado:")
print(" data:", col_data_bruta)
print(" valor_passagens:", col_valor_passagens)
print(" orgao:", col_orgao)
print(" destino:", col_destino)
print(" id_processo:", col_idproc)
print(" situacao:", col_situacao)

# 5) Converter a data (aceita dia/mês/ano; falhas viram NaT)
df["data_inicio"] = pd.to_datetime(df[col_data_bruta], format="%d/%m/%Y", errors="coerce")

# 6) Coluna Ano-Mês para séries mensais
df["data_inicio_ym"] = df["data_inicio"].dt.to_period("M").astype(str)

# 7) Garantir que valor_passagens é numérico
#    (se vier como texto com separador de milhar, converte)
if df[col_valor_passagens].dtype == "object":
    df[col_valor_passagens] = (
        df[col_valor_passagens].astype(str)
        .str.replace(".", "", regex=False)   # remove separador de milhar
        .str.replace(",", ".", regex=False)  # vírgula -> ponto
        .astype(float)
    )

# 8) Conferência rápida
display(df[[col_data_bruta, "data_inicio", "data_inicio_ym"]].head())
display(df[col_valor_passagens].describe())

In [ ]:
# Converter a coluna de data para datetime (dia/mês/ano)
# OBS: use as variáveis descobertas na célula anterior (col_data_bruta,
# col_valor_passagens). Os nomes originais com acento já foram normalizados.
df["data_inicio"] = pd.to_datetime(
    df[col_data_bruta], format="%d/%m/%Y", errors="coerce"
)

# Criar coluna Ano-Mês (AAAA-MM) para séries mensais
df["data_inicio_ym"] = df["data_inicio"].dt.to_period("M").astype(str)

# Conferir rapidamente
display(df[[col_data_bruta, "data_inicio", "data_inicio_ym"]].head())

# Estatísticas descritivas gerais
display(df.describe(include="all"))

# Estatísticas de uma coluna específica (valor das passagens)
display(df[col_valor_passagens].describe())

# Quantidade de nulos por coluna
df.isna().sum()

In [ ]:
# Histograma dos valores de passagens
plt.hist(df[col_valor_passagens].dropna(), bins=30)
plt.title("Distribuição dos valores de passagens")
plt.xlabel("Valor (R$)")
plt.ylabel("Frequência")
plt.show()

# Filtrar passagens entre 200 e 5000 (como no exemplo em R)
filtro = (df[col_valor_passagens] >= 200) & (df[col_valor_passagens] <= 5000)
passagens_filtradas = df.loc[filtro, col_valor_passagens]

plt.hist(passagens_filtradas.dropna(), bins=30)
plt.title("Valores de passagens (filtro: 200 a 5000)")
plt.xlabel("Valor (R$)")
plt.ylabel("Frequência")
plt.show()

# Boxplots (geral e filtrado)
plt.boxplot(df[col_valor_passagens].dropna())
plt.title("Boxplot - Valor das passagens (geral)")
plt.ylabel("Valor (R$)")
plt.show()

plt.boxplot(passagens_filtradas.dropna())
plt.title("Boxplot - Valor das passagens (200 a 5000)")
plt.ylabel("Valor (R$)")
plt.show()

# Desvio-padrão
print("Desvio-padrão dos valores de passagens:", df[col_valor_passagens].std())

# Distribuição da Situação (contagem e %)
print(df[col_situacao].value_counts())
print((df[col_situacao].value_counts(normalize=True) * 100).round(2))

In [ ]:
# Função para normalizar nomes de colunas
# (sem acento, minúsculas, _ no lugar de símbolos)
def normalizar_coluna(s):
    s = str(s)
    s = unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode("ascii")
    s = re.sub(r"[^\w]+", "_", s)
    s = re.sub(r"_+", "_", s)
    return s.strip("_").lower()

df.columns = [normalizar_coluna(c) for c in df.columns]
print("Colunas normalizadas:", list(df.columns))

# Função para achar coluna por "pistas"
def achar_coluna(palavras):
    for c in df.columns:
        if all(p in c for p in palavras):
            return c
    return None

col_valor = achar_coluna(["valor", "passag"])                    # valor_passagens
col_data  = achar_coluna(["periodo", "data", "inicio"]) or achar_coluna(["data", "inicio"])
col_orgao = achar_coluna(["nome", "orgao", "superior"]) or achar_coluna(["orgao"])  # nome_do_orgao_superior
col_dest  = achar_coluna(["destino"])                            # destinos
col_id    = achar_coluna(["identificador", "processo", "viagem"])

print("Mapeamento:")
print("  valor =", col_valor)
print("  data  =", col_data)
print("  orgao =", col_orgao)
print("  destino =", col_dest)
print("  id_proc =", col_id)

In [ ]:
# Converter data
df["data_inicio"] = pd.to_datetime(df[col_data], format="%d/%m/%Y", errors="coerce")

# Ano, mês e chave Ano-Mês
df["ano"]  = df["data_inicio"].dt.year
df["mes"]  = df["data_inicio"].dt.month
df["ano_mes"] = df["data_inicio"].dt.to_period("M").astype(str)

# Garantir que o valor das passagens é numérico
if df[col_valor].dtype == "object":
    df[col_valor] = (
        df[col_valor].astype(str)
        .str.replace(".", "", regex=False)   # separador de milhar
        .str.replace(",", ".", regex=False)  # vírgula -> ponto decimal
        .astype(float)
    )

# Remover linhas sem valor ou data
df_modelo = df.dropna(subset=[col_valor, "data_inicio"]).copy()
print(df_modelo.shape)
df_modelo[[col_dest, col_orgao, col_valor, "data_inicio", "ano", "mes"]]

In [ ]:
# Escolha de features (categorias + tempo). Ajuste conforme seu dataset.
features_categoricas = [col_orgao, col_dest]
features_numericas   = ["ano", "mes"]

# Mantemos apenas as colunas necessárias
base = df_modelo[features_categoricas + features_numericas + [col_valor]].copy()

# ATENÇÃO: "destinos" tem milhares de valores distintos. Aplicar one-hot em
# todos eles criaria dezenas de milhares de colunas e estouraria a memória.
# Solução: manter as TOP_N categorias mais frequentes e agrupar o resto em "OUTROS".
TOP_N = 50   # TODO: experimente 20, 50, 100...
for col in features_categoricas:
    mais_frequentes = base[col].value_counts().nlargest(TOP_N).index
    base[col] = base[col].where(base[col].isin(mais_frequentes), "OUTROS")
    print(f"{col}: {base[col].nunique()} categorias após o agrupamento")

# One-hot encoding para categorias
base_dummies = pd.get_dummies(base, columns=features_categoricas, drop_first=True)

X = base_dummies.drop(columns=[col_valor])
y_reg = base_dummies[col_valor]   # alvo para regressão
print("X shape:", X.shape, "| y_reg shape:", y_reg.shape)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor, export_text
from sklearn.metrics import mean_absolute_error, r2_score

# Split treino/teste
X_train, X_test, y_train, y_test = train_test_split(
    X, y_reg, test_size=0.2, random_state=42
)

# Modelo (ajuste max_depth e min_samples_leaf)
reg = DecisionTreeRegressor(
    random_state=42,
    max_depth=10,          # TODO: experimente valores (ex.: 5, 10, 20)
    min_samples_leaf=20,   # TODO: experimente valores (ex.: 5, 10, 50)
)
reg.fit(X_train, y_train)

# Avaliação
pred = reg.predict(X_test)
mae = mean_absolute_error(y_test, pred)
r2  = r2_score(y_test, pred)
print(f"MAE: {mae:,.2f}")
print(f"R² : {r2:.3f}")

# Importância das features (Top 15)
imp = pd.Series(reg.feature_importances_, index=X.columns).sort_values(ascending=False).head(15)
display(imp)

# (Opcional) Ver a árvore em texto (se a árvore não for muito grande)
print(export_text(reg, feature_names=list(X.columns))[:2000])